# Task #2 - Multimodal ML – Housing Price Prediction Using Images + Tabular Data    

### `Objective`:    
Predict housing prices using both structured data and house images.    

### `Dataset`:    
Housing Sales Dataset + Custom Image Dataset (your own or any public source) 

`Instructions`:    
● Use CNNs to extract features from images    
● Combine extracted image features with tabular data    
● Train a model using both modalities    
● Evaluate performance using MAE and RMSE

----------------

## Problem Statement:
Housing prices depend on both measurable attributes (bedrooms, bathrooms, square footage) and visual factors (interior condition, layout, aesthetic appeal) that structured data alone can't capture. Relying on tabular features only misses valuable visual signals that influence real-world pricing.

## Goal:

Build a multimodal model that combines CNN-extracted image features with structured tabular data to predict housing prices, using a pretrained CNN (MobileNetV2) for image feature extraction and a regression model trained on the fused feature set, evaluated using MAE and RMSE.

In [38]:
!git clone https://github.com/emanhamed/Houses-dataset.git

fatal: destination path 'Houses-dataset' already exists and is not an empty directory.


In [39]:
import pandas as pd

columns = ["bedrooms", "bathrooms", "area", "zipcode", "price"]
df = pd.read_csv(
    "Houses-dataset/Houses Dataset/HousesInfo.txt",
    sep=" ", header=None, names=columns
)
print(df.head())

   bedrooms  bathrooms  area  zipcode   price
0         4        4.0  4053    85255  869500
1         4        3.0  3343    36372  865200
2         3        4.0  3923    85266  889000
3         5        5.0  4022    85262  910000
4         3        4.0  4116    85266  971226


In [40]:
df.isnull().sum()

,0
bedrooms,0
bathrooms,0
area,0
zipcode,0
price,0


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 535 entries, 0 to 534
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   bedrooms   535 non-null    int64  
 1   bathrooms  535 non-null    float64
 2   area       535 non-null    int64  
 3   zipcode    535 non-null    int64  
 4   price      535 non-null    int64  
dtypes: float64(1), int64(4)
memory usage: 21.0 KB


In [42]:
df.shape

(535, 5)

In [43]:
df = df.drop_duplicates()

In [44]:
df.shape

(530, 5)

In [45]:
df.head()

,bedrooms,bathrooms,area,zipcode,price
0,4,4.0,4053,85255,869500
1,4,3.0,3343,36372,865200
2,3,4.0,3923,85266,889000
3,5,5.0,4022,85262,910000
4,3,4.0,4116,85266,971226


In [46]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scale continuous features
scaler = MinMaxScaler()
df[["bedrooms", "bathrooms", "area"]] = scaler.fit_transform(df[["bedrooms", "bathrooms", "area"]])

# One-hot encode zipcode (categorical, even though it looks numeric)
df = pd.get_dummies(df, columns=["zipcode"])

In [47]:
df

,bedrooms,bathrooms,area,price,zipcode_36372,zipcode_60002,zipcode_60016,zipcode_60046,zipcode_62025,zipcode_62034,...,zipcode_93720,zipcode_93924,zipcode_94501,zipcode_94531,zipcode_94565,zipcode_94568,zipcode_95008,zipcode_95220,zipcode_96019,zipcode_98021
0,0.333333,0.500000,0.377392,869500,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0.333333,0.333333,0.297456,865200,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0.222222,0.500000,0.362756,889000,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,0.444444,0.666667,0.373902,910000,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0.222222,0.500000,0.384485,971226,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
530,0.444444,0.166667,0.153682,399900,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
531,0.333333,0.416667,0.994708,460000,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
532,0.222222,0.166667,0.147827,407000,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
533,0.333333,0.333333,0.181378,419000,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False


In [48]:
df["price_scaled"] = df["price"] / df["price"].max()

In [49]:
df.head(5)

,bedrooms,bathrooms,area,price,zipcode_36372,zipcode_60002,zipcode_60016,zipcode_60046,zipcode_62025,zipcode_62034,...,zipcode_93924,zipcode_94501,zipcode_94531,zipcode_94565,zipcode_94568,zipcode_95008,zipcode_95220,zipcode_96019,zipcode_98021,price_scaled
0,0.333333,0.500000,0.377392,869500,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0.148429
1,0.333333,0.333333,0.297456,865200,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0.147695
2,0.222222,0.500000,0.362756,889000,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0.151758
3,0.444444,0.666667,0.373902,910000,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0.155343
4,0.222222,0.500000,0.384485,971226,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,0.165795


In [50]:
import cv2
import numpy as np

def load_house_images(house_id, base_path="Houses-dataset/Houses Dataset"):
    image_types = ["bathroom", "bedroom", "frontal", "kitchen"]
    images = []
    for img_type in image_types:
        path = f"{base_path}/{house_id}_{img_type}.jpg"
        img = cv2.imread(path)
        img = cv2.resize(img, (64, 64))
        images.append(img)

    # Arrange into a 2x2 montage: top row [bathroom, bedroom], bottom row [frontal, kitchen]
    top_row = np.hstack([images[0], images[1]])
    bottom_row = np.hstack([images[2], images[3]])
    montage = np.vstack([top_row, bottom_row])
    return montage

In [51]:
# !pip install tensorflow

In [52]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model

# Load pretrained MobileNetV2 (lightweight, good for CPU/low-RAM machines), excluding its final classification layers
base_model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg", input_shape=(128, 128, 3))
base_model.trainable = False  # freeze weights -- we're only extracting features, not fine-tuning

def extract_image_features(montage_image):
    img = preprocess_input(montage_image.astype("float32"))
    img = np.expand_dims(img, axis=0)
    features = base_model.predict(img, verbose=0)
    return features.flatten()  # 1280-dim feature vector for MobileNetV2

In [53]:
image_features = []
for house_id in df.index + 1:  # dataset IDs typically start at 1
    montage = load_house_images(house_id)
    features = extract_image_features(montage)
    image_features.append(features)

image_features = np.array(image_features)
print(image_features.shape)  # (num_houses, 1280)

(530, 1280)


In [54]:
tabular_features = df.drop(columns=["price", "price_scaled"]).values

combined_features = np.hstack([tabular_features, image_features])
print(combined_features.shape)

(530, 1332)


In [55]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(
    combined_features, df["price"].values, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, random_state=42)

In [56]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"MAE: ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

MAE: $223,040.09
RMSE: $308,576.54


In [57]:
import numpy as np

# Pick one house from the test set
sample_idx = 5
sample_features = X_test[sample_idx].reshape(1, -1)
actual_price = y_test[sample_idx]

predicted_price = model.predict(sample_features)[0]

print(f"Actual price:    ${actual_price:,.2f}")
print(f"Predicted price: ${predicted_price:,.2f}")
print(f"Difference:      ${abs(actual_price - predicted_price):,.2f}")

Actual price:    $699,999.00
Predicted price: $706,841.95
Difference:      $6,842.95


# Final Results

### - MAE: $223,040.09
### - RMSE: $308,576.54
### - Spot-check on an individual house: actual price $699,999 vs. predicted $706,841.95 — 
### - a difference of just $6,842.95, showing the model can be highly accurate on individual cases even with a wider average error across the full test set